# S5.V2 完整联合数学核验
纯合成；不是业务增益评估。需要已经创建的新双虚拟机实验目录。

In [1]:
from pathlib import Path
import json, yaml
runtime = json.loads(Path(RUNTIME_PATH).read_text())
cfg = yaml.safe_load(Path(runtime["config"]).read_text())
print(json.dumps({"step": cfg["step_id"], "config_sha256": runtime["config_sha256"], "code_sha256": runtime["code_sha256"], "model_seeds": cfg["shared"]["model_seeds"], "order_seeds": cfg["diagnostic"]["order_seeds"], "data_seed": cfg["smoke"]["data_seed"], "git_sha": runtime["git_sha"], "synthetic_only": True}, indent=2))
from datetime import datetime, timezone
print(datetime.now(timezone.utc).isoformat())

{
  "step": "S5.V2",
  "config_sha256": "705abbeefcab558dace83c8520e63f0e6017f93ccbb5612f735a75d3fe301d0c",
  "code_sha256": {
    "functional_training.py": "030c8faa4b9b9f39134d4fec78d396f4b64845131ffe8ba3a7c8ac96ffd7ff49",
    "functional_metrics.py": "fa11465b1eba9bc65d8043c90610ee7b8189fe6835c64e33807550ca6d9344f3",
    "functional_tls.py": "6753d344878ffb52e6b53da84cdd200b71b123fa9b3f876b1d649e0baeb163b3",
    "functional_vm.py": "93243f252adc57f44d9abd1bf36c638b384e45ea52620df5f5fa30d966873b08",
    "active_diagnostic.py": "95c01445081330eb4790212e799cfa3d896482e71d7a3e009c07c9331a62f282",
    "portfolio_binary_control.py": "0c0f517ce2071398bfb1f9a4cc30e9b07cd15d011fc0e57dcc6a5368b82abc6f",
    "portfolio_math.py": "8e0b2c8861ba0d94347f68ba8d8ea63c1eaf2d0135845874eaa90221edd08bb0"
  },
  "model_seeds": [
    11,
    22,
    33,
    44,
    55
  ],
  "order_seeds": {
    "11": 101,
    "22": 102,
    "33": 103,
    "44": 104,
    "55": 105
  },
  "data_seed": 20261005,
  "git_sha"

In [2]:
from portfolio_binary_control import train_joint
result = train_joint(runtime)
print(json.dumps(result))

{"completed_training": "joint_secure_seed11_candidate0", "epoch_count": 12, "synthetic_only": true}
{"completed_training": "joint_secure_seed22_candidate0", "epoch_count": 12, "synthetic_only": true}
{"completed_training": "joint_secure_seed33_candidate0", "epoch_count": 12, "synthetic_only": true}
{"completed_training": "joint_secure_seed44_candidate0", "epoch_count": 12, "synthetic_only": true}
{"completed_training": "joint_secure_seed55_candidate0", "epoch_count": 12, "synthetic_only": true}
{"status": "passed", "synthetic_only": true, "receiver": "alice"}


In [3]:
if runtime["party"] == "alice":
    report = json.loads((Path(runtime["config"]).parent.parent / "results/joint_math_report.json").read_text())
    print(json.dumps({"status": report["status"], "summary": report["summary"], "scope": report["scope"]}, indent=2))
else:
    print("Bob 完成固定训练调度，无明文标签、概率或联合权重输出。")

{
  "status": "passed",
  "summary": [
    {
      "learning_rate": 0.1,
      "split": "train",
      "maximum": 0.0001354217529296875,
      "mean": 0.00011830329895019531,
      "seed_mean_ci_95": [
        0.00010991096496582031,
        0.00012775659561157227
      ],
      "seed_count": 5
    },
    {
      "learning_rate": 0.1,
      "split": "validation",
      "maximum": 0.00013208389282226562,
      "mean": 0.00012164115905761718,
      "seed_mean_ci_95": [
        0.00011680126190185546,
        0.00012798309326171874
      ],
      "seed_count": 5
    }
  ],
  "scope": "\u4ec5\u516c\u5f00\u7a0b\u5e8f\u751f\u6210\u7684\u4e8c\u5904\u7406\u7ec4/\u666e\u901a\u5206\u7c7b\u6570\u5b66\u63a7\u5236\uff1b\u65e7\u771f\u5b9e\u6570\u636e\u4e0e\u5c01\u5b58\u4e0d\u8bfb\uff1b\u4e0d\u4f5c\u5b9e\u9645\u4e1a\u52a1\u6307\u6807"
}


工程通过判据为所有配置误差不超过预设阈值。区间仅描述初始化/批次顺序变化下的数值误差，不是收益置信区间。纯合成明文参照严格受代码 capability 限制；不能据此声称生产安全。